# Full experiment – RNA language models for cold-start miRNA–disease prediction
Every experiment in the paper, run **in the foreground**: the output of each training run appears under the cell as it finishes.

**Before you start**
1. Either upload the release zip of this repository to Google Drive in **MyDrive/mirna_lm_full/** (renamed `mirna_lm_full_package.zip`), or clone the repository into that folder (`project/`). Put the raw files listed in `data/README.md` in `project/data/raw/`.
2. Runtime → Change runtime type → **CPU** (the code runs on CPU).
3. Keep this browser tab open while a cell is running.

| Part | What | Approx. time (2 CPU cores) |
|---|---|---|
| 1 | Quick check: rebuild tables/figures from the original results | 3 min |
| 2 | Cross-validation, all 3 settings, MLP + GNN, RNA-FM / classical / random features | 1.5–2 h |
| 3 | Cold-miRNA: ERNIE-RNA, RiNALMo, k-mer, alignment, fusion; standard + debiased training; 3 seeds | 2.5–3 h |
| 4 | IMC model (cross-validation) | 2 min |
| 5 | Temporal split inside HMDD v3.0 | 30–40 min |
| 6 | Prospective validation: train HMDD v3.0 → test new miRNAs in HMDD v4.0, 10 seeds | 1.5–2 h |
| 7 | All analyses, statistics, tables and figures | 5 min |
| 8 | Compare your numbers with the original results | 1 min |

You do not have to run everything in one go. Results are saved to Drive after every single run. If Colab disconnects, run **Setup** again, then the same part again: finished runs are skipped and it continues where it stopped. Lines start with `[1]` or `[2]`, which shows which of the two parallel jobs printed them.

## Setup (run every time you open the notebook)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/mirna_lm_full'
if not os.path.exists(f'{BASE}/project'):
    !cd "{BASE}" && unzip -q mirna_lm_full_package.zip
%cd {BASE}/project
!mkdir -p results
!pip -q install biopython tabulate
!python -c "import torch, numpy, pandas, sklearn; print('torch', torch.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)"
!nproc

## Part 1 – Quick check (3 min)
Rebuilds the tables and figures from the **original** raw results in a temporary folder, to confirm that everything is installed correctly.

In [ ]:
!rm -rf /content/quick && mkdir -p /content/quick && cp -r src data /content/quick/ && cp -r results_original /content/quick/results
%cd /content/quick
!python src/analyze.py && python src/analyze_temporal.py && python src/make_tables.py && python src/make_figures.py
%cd {BASE}/project

## Part 2 – Cross-validation in all three settings (1.5–2 h)
Random / cold-miRNA / cold-disease, MLP and GNN, features: classical similarity, RNA-FM (mature, hairpin, both), RNA-FM + classical, random. Seed 0.

In [ ]:
!NTHREADS=1 bash -c "(python src/experiment.py --epochs 300 --out results/raw_random.csv --settings random --seeds 0) 2>&1 | sed -u 's/^/[1] /' & (python src/experiment.py --epochs 300 --out results/raw_cold.csv --settings cold_mirna cold_disease --seeds 0) 2>&1 | sed -u 's/^/[2] /' & wait"

## Part 3 – Cold-miRNA: all language models and sequence baselines (2.5–3 h)
Job [1]: popularity-debiased training, 9 features, 3 seeds (also saves predictions for the homology analysis). Job [2]: standard training: first MLP + GNN for 4 features (seed 0), then MLP for 8 features (3 seeds).

In [ ]:
!NTHREADS=1 bash -c "(python src/experiment.py --epochs 300 --out results/raw_b3_degree.csv --settings cold_mirna --models mlp --features rnafm_both ernierna_both rinalmo_both kmer align random ernierna_both+align rnafm_both+align sim --seeds 0 1 2 --train_neg degree --save_preds results/preds_degree) 2>&1 | sed -u 's/^/[1] /' & (python src/experiment.py --epochs 300 --out results/raw_b2_uniform.csv --settings cold_mirna --models mlp gnn --features ernierna_both rinalmo_both kmer align --seeds 0 --train_neg uniform ; python src/experiment.py --epochs 300 --out results/raw_b3_uniform.csv --settings cold_mirna --models mlp --features rnafm_both ernierna_both rinalmo_both kmer align random ernierna_both+align rnafm_both+align --seeds 0 1 2 --train_neg uniform) 2>&1 | sed -u 's/^/[2] /' & wait"

## Part 4 – IMC model (2 min)

In [ ]:
!NTHREADS=2 python src/experiment.py --epochs 300 --out results/raw_imc.csv --settings cold_mirna --models imc --features rnafm_both ernierna_both rinalmo_both kmer align random ernierna_both+align rnafm_both+align --seeds 0 1 2 --train_neg uniform

## Part 5 – Temporal split inside HMDD v3.0 (30–40 min)
Train on associations first reported up to PMID 25,000,000 (~mid-2014). Test on 222 miRNAs first reported later, and on new associations of known miRNAs.

In [ ]:
!python src/temporal.py --out results/temporal_popularity.csv --popularity --features x --seeds 0 1 2 3 4
!NTHREADS=1 bash -c "(python src/temporal.py --epochs 300 --out results/temporal_degree.csv --features ernierna_both rnafm_both rinalmo_both align kmer random ernierna_both+align --train_neg degree --seeds 0 1 2 3 4) 2>&1 | sed -u 's/^/[1] /' & (python src/temporal.py --epochs 300 --out results/temporal_uniform.csv --features ernierna_both rnafm_both rinalmo_both align kmer random ernierna_both+align --train_neg uniform --seeds 0 1 2 3 4) 2>&1 | sed -u 's/^/[2] /' & wait"
!NTHREADS=2 python src/temporal.py --epochs 300 --out results/temporal_uniform.csv --features ernierna_both rnafm_both rinalmo_both align kmer random ernierna_both+align --models imc --train_neg uniform --seeds 0 1 2 3 4

## Part 6 – Prospective validation HMDD v3.0 → v4.0 (1.5–2 h)
Train on everything up to 2018 (HMDD v3.0). Test on the 443 miRNAs first reported in HMDD v4.0 (2019–2023), and on 14,009 new associations of known miRNAs. 10 seeds.

In [ ]:
!python src/temporal.py --data data/processed_v34 --cut 29845432 --out results/prosp_popularity.csv --popularity --features x --seeds 0 1 2 3 4 5 6 7 8 9
!NTHREADS=1 bash -c "(python src/temporal.py --epochs 300 --data data/processed_v34 --cut 29845432 --out results/prosp_degree.csv --features ernierna_both rnafm_both rinalmo_both align kmer random ernierna_both+align --train_neg degree --seeds 0 1 2 3 4 5 6 7 8 9) 2>&1 | sed -u 's/^/[1] /' & (python src/temporal.py --epochs 300 --data data/processed_v34 --cut 29845432 --out results/prosp_uniform.csv --features ernierna_both rnafm_both rinalmo_both align kmer random ernierna_both+align --train_neg uniform --seeds 0 1 2 3 4 5 6 7 8 9) 2>&1 | sed -u 's/^/[2] /' & wait"
!NTHREADS=2 python src/temporal.py --epochs 300 --data data/processed_v34 --cut 29845432 --out results/prosp_uniform.csv --features ernierna_both rnafm_both rinalmo_both align kmer random ernierna_both+align --models imc --train_neg uniform --seeds 0 1 2 3 4 5 6 7 8 9

## Part 7 – Analyses, statistics, tables and figures (5 min)

In [ ]:
!python src/sanity_baselines.py data/processed
!python src/analyze.py
!python src/extra_analysis.py
!python src/build_confounders.py data/raw data/processed
!python src/confounder_analysis.py
!python src/covariate_probe.py
!python src/coreport.py data/raw
!python src/analyze_temporal.py
!python src/make_tables.py
!python src/make_figures.py

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 200)
s = pd.read_csv('results/summary_all.csv')
display(s[(s.setting=='cold_mirna')].pivot_table(index=['model','train_neg','feature'], columns='neg', values='auc_mean').round(3))
t = pd.read_csv('results/temporal_summary.csv')
display(t[t.neg.str.startswith('tcold')].pivot_table(index=['split','model','train_neg','feature'], columns='neg', values='auc_mean').round(3))
for f in ['popularity_encoding', 'popularity_confounders', 'covariate_probe', 'coreporting']:
    print(f); display(pd.read_csv(f'results/{f}.csv'))
for f in ['fig1_cold_mirna_auc', 'fig2_popularity_encoding', 'fig3_homology_strata', 'fig4_prospective']:
    display(Image(f'figures/{f}.png', width=750))

## Part 8 – Compare with the original results
Small differences (about 0.01 AUROC) are normal because of CPU and library versions. The ranking and conclusions should be the same.

In [ ]:
!python src/compare_all.py

## Part 9 – Robustness analyses for revision (6–9 h; resumable)
Exact degree-preserving negatives, family-level hold-out (name family, seed family, homology cluster), false-negative check, prospective sensitivity (aliases, novelty strata) and miRNA-level bootstrap CIs. Every run is skipped if it is already in the output CSV, so if Colab disconnects just run the same cell again.

**9a – families and quick checks (5 min)**

In [ ]:
!python src/build_families.py data/raw data/processed
!python src/build_families.py data/raw data/processed_v34
!python src/false_negatives.py
!python src/prospective_checks.py
!python src/novelty_compare.py
!python src/bootstrap_ci.py results/preds_degree "cold_mirna_{{neg}}_mlp_{{feat}}_s{{seed}}_f{{fold}}.npz" results/bootstrap_cv_debiased 1000

**9b – exact negatives with random miRNA folds (MLP ~1.5 h, IMC 5 min)**

In [ ]:
!NTHREADS=2 python src/robust.py --split kfold --train_neg degree --features ernierna_both rnafm_both rinalmo_both align kmer random --out results/robust_kfold_degree.csv --save_preds results/preds_robust
!NTHREADS=2 python src/robust.py --split kfold --models imc --train_neg uniform --features ernierna_both rnafm_both rinalmo_both align kmer random --out results/robust_kfold_imc.csv

**9c – family-level hold-out (MLP ~1 h per grouping, IMC 5 min each)**

In [ ]:
G = 'data/processed/mirna_family.csv'
for g, d in [('homology70', 'hom70'), ('name_family', 'namefam'), ('seed_family', 'seedfam')]:
    !NTHREADS=2 python src/robust.py --split group --groups {G} --group_col {g} --train_neg degree --features ernierna_both rnafm_both align random --out results/robust_{g}.csv --save_preds results/preds_robust_{d}
    !NTHREADS=2 python src/robust.py --split group --groups {G} --group_col {g} --models imc --train_neg uniform --features ernierna_both rnafm_both rinalmo_both align kmer random --out results/robust_{g}_imc.csv

**9d – prospective predictions (MLP, debiased, 10 seeds; ~1.5 h)**

In [ ]:
!NTHREADS=2 python src/temporal.py --data data/processed_v34 --cut 29845432 --out results/prosp_preds_degree.csv --features ernierna_both rnafm_both rinalmo_both align random --train_neg degree --seeds 0 1 2 3 4 5 6 7 8 9 --save_preds results/preds_prosp

**9e – statistics and tables (15 min)**

In [ ]:
!python src/analyze_robust.py
!python src/prospective_robust.py
!FEATS="ernierna_both rnafm_both rinalmo_both align kmer random" python src/bootstrap_ci.py results/preds_robust "kfold_{{neg}}_mlp_{{feat}}_degree_s{{seed}}_f{{fold}}.npz" results/bootstrap_robust_kfold 1000
for g, d in [('homology70', 'hom70'), ('name_family', 'namefam'), ('seed_family', 'seedfam')]:
    !FEATS="ernierna_both rnafm_both align random" python src/bootstrap_ci.py results/preds_robust_{d} "group_{{neg}}_mlp_{{feat}}_degree_s{{seed}}_f{{fold}}.npz" results/bootstrap_robust_{g} 1000